In [ ]:
import json, os , sys
from datasets import Dataset

sys.path.append(r"..\sales_marketing_rag")

from app.rag import answer
 
gold = [json.loads(l) for l in open(r"...\sales_marketing_rag\data\golden\eval_set.jsonl")]
records = []
for g in gold:
    r = answer(g["question"])
    records.append({
        "user_input": g["question"],
        "response": r["answer"],
        "retrieved_contexts": r["contexts"],
        "reference": g["ground_truth"],
    })

In [ ]:
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (LLMContextPrecisionWithReference, LLMContextRecall,
                            Faithfulness, ResponseRelevancy)

judge_llm = LangchainLLMWrapper(ChatOllama(model=os.environ["JUDGE_MODEL"], temperature=0))
judge_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=os.environ["EMBED_MODEL"]))

ds = EvaluationDataset.from_list(records)
result = evaluate(
    dataset=ds,
    metrics=[LLMContextPrecisionWithReference(), LLMContextRecall(),
             Faithfulness(), ResponseRelevancy()],
    llm=judge_llm, embeddings=judge_emb,
)

print(result)
result.to_pandas().to_csv(r"...\sales_marketing_rag\data\output\eval_results.csv", index=False)

C:\Users\brand\AppData\Local\Temp\ipykernel_6344\3277416223.py:6: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (LLMContextPrecisionWithReference, LLMContextRecall,
C:\Users\brand\AppData\Local\Temp\ipykernel_6344\3277416223.py:6: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (LLMContextPrecisionWithReference, LLMContextRecall,
C:\Users\brand\AppData\Local\Temp\ipykernel_6344\3277416223.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example

{'llm_context_precision_with_reference': nan, 'context_recall': nan, 'faithfulness': nan, 'answer_relevancy': nan}
